# Compiling systems from Latković et al. 2021


ADS link: https://ui.adsabs.harvard.edu/abs/2021ApJS..254...10L/abstract

Authors: Latković, Olivera search by orcid ; Čeki, Atila search by orcid ; Lazarević, Sanja 

"The compilation includes nearly 700 individually investigated objects from over 450 distinct publications."

There is a VizieR object: https://ui.adsabs.harvard.edu/abs/2021yCat..22540010L/abstract
and their own website https://wumacat.aob.rs/Downloads

We have downloaded `WUMaCat.csv` into `data/from_others` to extract the names and masses into our main catalog

In [1]:
import json
import math
from pathlib import Path
import math

from astroquery.simbad import Simbad
from astroquery.vizier import Vizier
import pandas as pd

In [2]:
# Load the source catalog from Latkovic + 2021 and inspect core columns
PROJECT_ROOT = Path("../..").resolve()
wuma_csv = PROJECT_ROOT / "data" / "from_others" / "WUMaCat.csv"

df = pd.read_csv(wuma_csv)
print(f"Loaded {len(df)} rows from {wuma_csv}")
display(df[["Name", "Bibcode", "P", "M1", "M2", "Type", "QT"]].head(10))

Loaded 688 rows from /Users/liekevanson/Documents/Projects/post_mt_review/data/from_others/WUMaCat.csv


,Name,Bibcode,P,M1,M2,Type,QT
0,1SWASP J003033.05+574347.6,2020AJ....159..189L,0.226618,0.790,0.380,W,PH
1,1SWASP J011732.10+525204.9,2018NewA...59....8S,0.223960,0.790,0.320,A,PH
2,1SWASP J015100.23-100524.2,2015AJ....150..117Q,0.214500,NaN,NaN,W,PH
3,1SWASP J024148.62+372848.3,2015NewA...41...22J,0.219751,NaN,NaN,W,PH
4,1SWASP J030749.87-365201.7,2018Ap&SS.363...15L,0.226671,NaN,NaN,W,PH
5,1SWASP J031700.67+190839.6,2020AJ....159..189L,0.225640,0.750,0.190,W,PH
6,1SWASP J034501.24+493659.9,2019AJ....157...73K,0.376532,0.654,0.275,W,SP
7,1SWASP J044132.96+440613.7,2018NewA...62...41K,0.228155,0.703,0.448,A,PH
8,1SWASP J050904.45-074144.4,2019MNRAS.485.4588L,0.229575,NaN,NaN,A,PH
9,1SWASP J052926.88+461147.5,2018NewA...62...41K,0.226642,0.804,0.331,A,PH


In [3]:
# Fetch coordinates from VizieR and coordinate uncertainties from SIMBAD
df = pd.read_csv(wuma_csv)

# VizieR exposes the catalog positions directly via _RA/_DE, keyed by system name.
vizier = Vizier(row_limit=-1)
vizier_table = vizier.get_catalogs("J/ApJS/254/10")[0]
vizier_df = vizier_table.to_pandas()[["Name", "SimbadName", "_RA", "_DE"]].copy()
vizier_df = vizier_df.rename(columns={"_RA": "ra_deg", "_DE": "dec_deg"})
vizier_df["Name"] = vizier_df["Name"].astype(str).str.strip()
vizier_df["SimbadName"] = vizier_df["SimbadName"].astype(str).str.strip()

# SIMBAD gives the coordinate error ellipse, which we use to estimate RA/Dec errors.
simbad = Simbad()
simbad.add_votable_fields("ra", "dec", "coo_err_maj", "coo_err_min", "coo_err_angle")

query_names = [name for name in vizier_df["SimbadName"].dropna().astype(str).unique() if name.strip()]
simbad_result = simbad.query_objects(query_names)

coord_err_df = simbad_result.to_pandas()[["user_specified_id", "coo_err_maj", "coo_err_min", "coo_err_angle"]].copy()
coord_err_df = coord_err_df.rename(columns={"user_specified_id": "SimbadName"})
coord_err_df["SimbadName"] = coord_err_df["SimbadName"].astype(str).str.strip()

df["Name"] = df["Name"].astype(str).str.strip()
df = df.merge(vizier_df, on="Name", how="left")
df = df.merge(coord_err_df, on="SimbadName", how="left")

def ellipse_to_radec_errors(row):
    if pd.isna(row.get("coo_err_maj")) or pd.isna(row.get("coo_err_min")) or pd.isna(row.get("coo_err_angle")) or pd.isna(row.get("dec_deg")):
        return pd.Series([None, None])

    major_mas = float(row["coo_err_maj"])
    minor_mas = float(row["coo_err_min"])
    angle_deg = float(row["coo_err_angle"])
    dec_deg = float(row["dec_deg"])

    # Project the SIMBAD error ellipse onto the RA and Dec axes.
    theta = math.radians(angle_deg)
    sigma_ra_sky_mas = math.sqrt((major_mas ** 2) * (math.sin(theta) ** 2) + (minor_mas ** 2) * (math.cos(theta) ** 2))
    sigma_dec_mas = math.sqrt((major_mas ** 2) * (math.cos(theta) ** 2) + (minor_mas ** 2) * (math.sin(theta) ** 2))

    # Convert mas to degrees; RA also needs the cos(dec) correction.
    cos_dec = math.cos(math.radians(dec_deg))
    if abs(cos_dec) < 1e-12:
        ra_err_deg = None
    else:
        ra_err_deg = sigma_ra_sky_mas / (1000.0 * 3600.0 * cos_dec)

    dec_err_deg = sigma_dec_mas / (1000.0 * 3600.0)
    return pd.Series([ra_err_deg, dec_err_deg])

df[["ra_err_deg", "dec_err_deg"]] = df.apply(ellipse_to_radec_errors, axis=1)
print(f"VizieR coordinates merged for {df['ra_deg'].notna().sum()} / {len(df)} rows")
print(f"SIMBAD coordinate errors merged for {df['ra_err_deg'].notna().sum()} / {len(df)} rows")
display(df[["Name", "SimbadName", "ra_deg", "dec_deg", "ra_err_deg", "dec_err_deg"]].head(10))

VizieR coordinates merged for 688 / 688 rows
SIMBAD coordinate errors merged for 685 / 688 rows


,Name,SimbadName,ra_deg,dec_deg,ra_err_deg,dec_err_deg
0,1SWASP J003033.05+574347.6,1SWASP J003033.05+574347.6,7.63795,57.72980,6.555379e-09,4.000000e-09
1,1SWASP J011732.10+525204.9,1SWASP J011732.10+525204.9,19.38376,52.86817,5.798052e-09,2.888889e-09
2,1SWASP J015100.23-100524.2,1SWASP J015100.23-100524.2,27.75098,-10.09008,5.276046e-09,4.583333e-09
3,1SWASP J024148.62+372848.3,1SWASP J024148.62+372848.3,40.45264,37.48001,7.980856e-09,5.111111e-09
4,1SWASP J030749.87-365201.7,1SWASP J030749.87-365201.7,46.95804,-36.86726,5.069270e-09,5.305555e-09
5,1SWASP J031700.67+190839.6,1SWASP J031700.67+190839.6,49.25289,19.14429,2.472872e-08,2.066667e-08
6,1SWASP J034501.24+493659.9,1SWASP J034501.24+493659.9,56.25522,49.61663,1.440554e-08,8.000000e-09
7,1SWASP J044132.96+440613.7,1SWASP J044132.96+440613.7,70.38732,44.10375,1.899353e-08,9.527778e-09
8,1SWASP J050904.45-074144.4,1SWASP J050904.45-074144.4,77.26852,-7.69563,3.363628e-09,2.777778e-09
9,1SWASP J052926.88+461147.5,1SWASP J052926.88+461147.5,82.36220,46.19649,9.952347e-09,5.083333e-09


In [4]:
# Helper functions for schema-compliant transformation
def to_float(value):
    if pd.isna(value):
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None

def triplet(val, err=None):
    if val is None:
        return [None, None, None]
    if err is None:
        return [None, val, None]
    return [err, val, err]


def build_detection_methods(row):
    methods = ["EB"]

    qt = str(row.get("QT", "") or "").strip().upper()
    if qt == "SP":
        methods.append("RV")

    solver = str(row.get("Solver", "") or "").strip().upper()
    if solver in {"WD", "PHOEBE", "BYNSYN", "ROCHE", "LIGHT2", "DC", "BINARY MAKER"}:
        methods.append("LC")

    return methods


def transform_row(row):
    p = to_float(row.get("P"))
    m1_raw = to_float(row.get("M1"))
    m2_raw = to_float(row.get("M2"))

    ra_deg = to_float(row.get("ra_deg"))
    dec_deg = to_float(row.get("dec_deg"))
    ra_err_deg = to_float(row.get("ra_err_deg"))
    dec_err_deg = to_float(row.get("dec_err_deg"))

    # Enforce: M2 is presumed donor and lower mass component
    m1 = None
    m2 = None
    swapped = False
    if m1_raw is not None and m2_raw is not None:
        m1 = max(m1_raw, m2_raw)
        m2 = min(m1_raw, m2_raw)
        swapped = m1_raw < m2_raw
    elif m1_raw is not None:
        m1 = m1_raw
    elif m2_raw is not None:
        m2 = m2_raw

    notes = [
        " source_file: from_others/WUMaCat.csv ",
        " M2 set to lower current mass ",
    ]
    # if ra_deg is not None and dec_deg is not None:
        # notes.append("RA/Dec from VizieR J/ApJS/254/10")
    if ra_err_deg is not None and dec_err_deg is not None:
        notes.append("coordinate err derived from SIMBAD error ellipse")
    if swapped:
        notes.append("Input masses were swapped to enforce donor as M2")

    return {
        "System Name": str(row.get("Name", "") or "").strip(),
        "RA": triplet(ra_deg, ra_err_deg),
        "Dec": triplet(dec_deg, dec_err_deg),
        "Period": triplet(p),
        "Eccentricity": [None, 0.0, None],
        "M1": triplet(m1),
        "M2": triplet(m2),
        "Mass Function": [None, None, None],
        "M1_sin3i": [None, None, None],
        "M2_sin3i": [None, None, None],
        "evol_type_1": "MS",
        "evol_type_2": "MS",
        "obs_type_1": None,
        "obs_type_2": None,
        "system_class": "WUMa binary",
        "Detection Method": build_detection_methods(row),
        "Reference": ["2021ApJS..254...10L"] + ([str(row.get("Bibcode")).strip()] if pd.notna(row.get("Bibcode")) and str(row.get("Bibcode")).strip() else []),
        "Notes": "e=0 baked in model ".join(notes),
        "Simbad": None,
    }

In [5]:
# Apply transformation and inspect intermediate output
records = [transform_row(row) for _, row in df.iterrows() if str(row.get("Name", "") or "").strip()]
print(f"Transformed rows: {len(records)}")

preview = pd.DataFrame({
    "System Name": [r["System Name"] for r in records[:10]],
    "M1": [r["M1"][1] for r in records[:10]],
    "M2": [r["M2"][1] for r in records[:10]],
    "system_class": [r["system_class"] for r in records[:10]],
    "Reference_count": [len(r["Reference"]) for r in records[:10]],
})
display(preview)


Transformed rows: 688


,System Name,M1,M2,system_class,Reference_count
0,1SWASP J003033.05+574347.6,0.790,0.380,WUMa binary,2
1,1SWASP J011732.10+525204.9,0.790,0.320,WUMa binary,2
2,1SWASP J015100.23-100524.2,NaN,NaN,WUMa binary,2
3,1SWASP J024148.62+372848.3,NaN,NaN,WUMa binary,2
4,1SWASP J030749.87-365201.7,NaN,NaN,WUMa binary,2
5,1SWASP J031700.67+190839.6,0.750,0.190,WUMa binary,2
6,1SWASP J034501.24+493659.9,0.654,0.275,WUMa binary,2
7,1SWASP J044132.96+440613.7,0.703,0.448,WUMa binary,2
8,1SWASP J050904.45-074144.4,NaN,NaN,WUMa binary,2
9,1SWASP J052926.88+461147.5,0.804,0.331,WUMa binary,2


In [6]:
# 4) Probe SIMBAD for coordinates and coordinate uncertainties
from astroquery.simbad import Simbad

simbad = Simbad()
simbad.add_votable_fields("ra(d)", "dec(d)", "coo_err_maj", "coo_err_min", "coo_err_angle")

probe_names = ["44 Boo", "AA UMa", "1SWASP J003033.05+574347.6", "ASAS J083241+2332.4"]
probe_result = simbad.query_objects(probe_names)
probe_result

/var/folders/5d/vcxrsh5975l7d5n8t7vvc5pr0000gn/T/ipykernel_99599/4020879615.py:5: DeprecationWarning: 'dec(d)' has been renamed 'dec'. You'll see it appearing with its new name in the output table
  simbad.add_votable_fields("ra(d)", "dec(d)", "coo_err_maj", "coo_err_min", "coo_err_angle")
/var/folders/5d/vcxrsh5975l7d5n8t7vvc5pr0000gn/T/ipykernel_99599/4020879615.py:5: DeprecationWarning: 'ra(d)' has been renamed 'ra'. You'll see it appearing with its new name in the output table
  simbad.add_votable_fields("ra(d)", "dec(d)", "coo_err_maj", "coo_err_min", "coo_err_angle")


main_id,ra,dec,coo_err_maj,coo_err_min,coo_err_angle,coo_wavelength,coo_bibcode,user_specified_id,object_number_id
,deg,deg,mas,mas,deg,,,,
object,float64,float64,float32,float32,int16,str1,object,object,int64
* i Boo,225.94706519512025,47.65406187958964,14.673812,12.653285,90,O,2007A&A...474..653V,44 Boo,1
V* AA UMa,146.74702048794,45.76566349130001,0.011,0.0085,90,O,2020yCat.1350....0G,AA UMa,2
1SWASP J003033.05+574347.6,7.63794563281,57.72980446758999,0.0126,0.0144,90,O,2020yCat.1350....0G,1SWASP J003033.05+574347.6,3
ASAS J083241+2332.4,128.17047421543,23.540506173179995,0.0616,0.0427,90,O,2020yCat.1350....0G,ASAS J083241+2332.4,4


In [7]:
# Inspect why SIMBAD uncertainty matches are sparse
print(vizier_df[["Name", "SimbadName"]].head(5).to_string(index=False))
print()
print(coord_err_df.head(5).to_string(index=False))
print()
print(type(vizier_df.loc[0, "SimbadName"]))
print(type(coord_err_df.loc[0, "SimbadName"]))
print()
vizier_names = set(vizier_df["SimbadName"].dropna().astype(str))
coord_names = set(coord_err_df["SimbadName"].dropna().astype(str))
print("Intersection size:", len(vizier_names & coord_names))

                      Name                 SimbadName
1SWASP J003033.05+574347.6 1SWASP J003033.05+574347.6
1SWASP J011732.10+525204.9 1SWASP J011732.10+525204.9
1SWASP J015100.23-100524.2 1SWASP J015100.23-100524.2
1SWASP J024148.62+372848.3 1SWASP J024148.62+372848.3
1SWASP J030749.87-365201.7 1SWASP J030749.87-365201.7

                SimbadName  coo_err_maj  coo_err_min  coo_err_angle
1SWASP J003033.05+574347.6       0.0126       0.0144             90
1SWASP J011732.10+525204.9       0.0126       0.0104             90
1SWASP J015100.23-100524.2       0.0187       0.0165             90
1SWASP J024148.62+372848.3       0.0228       0.0184             90
1SWASP J030749.87-365201.7       0.0146       0.0191             90

<class 'str'>
<class 'str'>

Intersection size: 688


In [8]:
# Save transformed records
output_json = PROJECT_ROOT / "data" / "result_tables" / "raw_json" / "WUMaCat.json"
output_json.parent.mkdir(parents=True, exist_ok=True)
with open(output_json, "w", encoding="utf-8") as f:
    f.write("[\n")
    for i, system in enumerate(records):
        line = json.dumps(system, separators=(",", ": "), ensure_ascii=False)
        f.write("  " + line)
        if i < len(records) - 1:
            f.write(",\n")
        else:
            f.write("\n")
    f.write("]\n")

print(f"Saved {len(records)} records to {output_json}")


Saved 688 records to /Users/liekevanson/Documents/Projects/post_mt_review/data/result_tables/raw_json/WUMaCat.json
